In [1]:
# !pip install openai

In [3]:
from openai import OpenAI
import numpy as np

client = OpenAI()

phrase_1_similar = "What is the capital of France?"
phrase_2_similar = "What is the capital of Germany?"
phrase_3_dissimilar = "Oceans have a lot of water."

def get_embedding_vector(response):
    # For OpenAI v1: response is a CreateEmbeddingResponse, use .data[0].embedding
    if hasattr(response, "data"):
        return response.data[0].embedding
    elif hasattr(response, "embedding"):
        return response.embedding
    else:
        # Already a vector
        return response

embedding_1_similar_resp = client.embeddings.create(
    input=phrase_1_similar,
    model="text-embedding-3-small"
)
embedding_2_similar_resp = client.embeddings.create(
    input=phrase_2_similar,
    model="text-embedding-3-small"
)
embedding_3_dissimilar_resp = client.embeddings.create(
    input=phrase_3_dissimilar,
    model="text-embedding-3-small"
)

# Actually get numpy array embedding vectors
embedding_1_similar = np.array(get_embedding_vector(embedding_1_similar_resp))
embedding_2_similar = np.array(get_embedding_vector(embedding_2_similar_resp))
embedding_3_dissimilar = np.array(get_embedding_vector(embedding_3_dissimilar_resp))

def cosine_distance(embedding_1, embedding_2):
    return np.dot(embedding_1, embedding_2) / (np.linalg.norm(embedding_1) * np.linalg.norm(embedding_2))

print(cosine_distance(embedding_1_similar, embedding_2_similar))
print(cosine_distance(embedding_1_similar, embedding_3_dissimilar))
print(cosine_distance(embedding_2_similar, embedding_3_dissimilar))

import pandas as pd
from IPython.display import display, HTML

# Prepare labels and vectors
phrases = [
    ("Phrase 1 (France)", embedding_1_similar),
    ("Phrase 2 (Germany)", embedding_2_similar),
    ("Phrase 3 (Oceans)", embedding_3_dissimilar),
]

phrase_names = [label for label, _ in phrases]
vectors = [vec for _, vec in phrases]

# Compute cosine similarity matrix
matrix = np.zeros((len(vectors), len(vectors)))
for i in range(len(vectors)):
    for j in range(len(vectors)):
        matrix[i][j] = cosine_distance(vectors[i], vectors[j])

# Make DataFrame
df = pd.DataFrame(matrix, index=phrase_names, columns=phrase_names)

# Style DataFrame for HTML
styled = (
    df.style
    .background_gradient(cmap="YlGnBu")
    .format("{:.4f}")
    .set_caption("Cosine Similarity Matrix")
)

display(HTML("<h3>Cosine Similarity Results</h3>"))
display(styled)

0.5393585194552265
0.0453919801980672
0.03913983297160628


,Phrase 1 (France),Phrase 2 (Germany),Phrase 3 (Oceans)
Phrase 1 (France),1.0000,0.5394,0.0454
Phrase 2 (Germany),0.5394,1.0000,0.0391
Phrase 3 (Oceans),0.0454,0.0391,1.0000
